## Video Generation Notebook
### For Tracker Data
#### Helper Functions for creating mp4s and GIFs of tracker images

In [2]:
import os
import re
import cv2
import yaml
import glob
import json
from pathlib import Path
from PIL import Image
import numpy as np
from tkinter import filedialog
import tkinter as tk
from tqdm import tqdm  # For progress bars (install with pip install tqdm if needed)
from pathlib import Path
print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")
np.set_printoptions(precision=2, suppress=True)  # neat printing


OpenCV version: 4.6.0
NumPy version: 1.26.4


In [3]:
def create_gif_pillow(input_folder, output_path, duration=100):
    """
    Create GIF animation using PIL/Pillow
    
    Args:
        input_folder: Path to folder containing images
        output_path: Output GIF file path
        duration: Duration between frames in milliseconds
    """
    # Get all jpg files and sort them numerically
    pattern = os.path.join(input_folder, "*.jpg")
    image_files = glob.glob(pattern)
    
    # Sort by the first number in filename (1_0, 2_1, 3_2, etc.)
    image_files.sort(key=lambda x: int(os.path.basename(x).split('_')[0]))
    
    if not image_files:
        print("No JPG files found in the specified folder!")
        return
    
    # Load images
    images = []
    for file in image_files:
        img = Image.open(file)
        images.append(img)
    
    # Save as GIF
    images[0].save(
        output_path,
        save_all=True,
        append_images=images[1:],
        duration=duration,
        loop=0  # 0 means infinite loop
    )
    print(f"GIF saved to: {output_path}")

def create_mp4_opencv(input_folder, output_path, fps=30):
    """
    Create MP4 video using OpenCV
    
    Args:
        input_folder: Path to folder containing images
        output_path: Output MP4 file path
        fps: Frames per second
    """
    pattern = os.path.join(input_folder, "*.jpg")
    image_files = glob.glob(pattern)
    image_files.sort(key=lambda x: int(os.path.basename(x).split('_')[0]))
    
    if not image_files:
        print("No JPG files found!")
        return
    
    # Read first image to get dimensions
    first_image = cv2.imread(image_files[0])
    height, width, layers = first_image.shape
    
    # Define codec and create VideoWriter
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # Add each image to video
    for file in image_files:
        img = cv2.imread(file)
        video.write(img)
    
    # Release everything
    video.release()
    cv2.destroyAllWindows()
    print(f"MP4 saved to: {output_path}")

#### Example Use Case for One Folder

In [ ]:
# Your specific folder path
submodule = os.path.expanduser("~") + "/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/video/"
folder_name= "08_19_2025_11_31_41_Medium_Turtle_Track"
input_folder = submodule + folder_name + "/track/Tracker-result"
output_folder = submodule + folder_name + "/track/animations/"
os.makedirs(output_folder, exist_ok=True)
# Check if folder exists
if not os.path.exists(input_folder):
    print(f"Folder not found: {input_folder}")
    exit()

# Create different animation formats
print("Creating animations...")

# GIF using PIL (good for web sharing)
# create_gif_pillow(input_folder, "animation_pillow.gif", duration=100)

# MP4 using OpenCV (high quality video)
create_mp4_opencv(input_folder, output_folder + folder_name + ".mp4", fps=15)

print("All animations created successfully!")

#### Tracker Video Generation

In [ ]:
# Your specific folder path
submodule = os.path.expanduser("~") + "/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/video/"

    # Find all folders starting with "08_19_2025"
folders_to_process = []
if os.path.exists(submodule):
    for folder in os.listdir(submodule):
        if folder.startswith("08_19_2025") and os.path.isdir(os.path.join(submodule, folder)):
            folders_to_process.append(folder)

for folder_name in folders_to_process:
    print(f"Processing folder: {folder_name}")
    input_folder = submodule + folder_name + "/track/Tracker-result"
    output_folder = submodule + folder_name + "/track/animations/"
    os.makedirs(output_folder, exist_ok=True)
    # Check if folder exists
    if not os.path.exists(input_folder):
        print(f"Folder not found: {input_folder}")
        exit()

    # Create different animation formats
    print("Creating mp4...")
    # MP4 using OpenCV (high quality video)
    create_mp4_opencv(input_folder, output_folder + folder_name + ".mp4", fps=15)
    print("Creating gif...")
    # GIF using PIL (good for web sharing)
    create_gif_pillow(input_folder, "animation_pillow.gif", duration=100)

    print(f"All animations created successfully for {folder_name}!")


Processing folder: 08_19_2025_10_41_08
Creating mp4...
MP4 saved to: /home/flyingwhale/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/video/08_19_2025_10_41_08/track/animations/08_19_2025_10_41_08.mp4
Creating gif...
GIF saved to: animation_pillow.gif
All animations created successfully for 08_19_2025_10_41_08!
Processing folder: 08_19_2025_10_14_57_Stingray_Turtle
Creating mp4...
MP4 saved to: /home/flyingwhale/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/video/08_19_2025_10_14_57_Stingray_Turtle/track/animations/08_19_2025_10_14_57_Stingray_Turtle.mp4
Creating gif...
GIF saved to: animation_pillow.gif
All animations created successfully for 08_19_2025_10_14_57_Stingray_Turtle!
Processing folder: 08_19_2025_11_05_23
Creating mp4...
MP4 saved to: /home/flyingwhale/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/video/08_19_2025_11_05_23/track/animations/08_19_2025_11_05_23.mp4
Creating gif...
GIF saved to: animation_pillow.gif
All animations created successfull

: 

### Video Generation for Left/Right Images
#### Helper Functions

In [4]:
def natural_sort_key(s):
    """
    Sort strings containing numbers naturally (e.g., frame1.npy, frame2.npy, ..., frame10.npy)
    """
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]

def create_video_from_images(image_folder, output_path, fps=15, extension="jpg"):
    """
    Create a video from a folder of images
    
    Args:
        image_folder (str): Path to folder containing images
        output_path (str): Path where to save the output video
        fps (int): Frames per second for the output video
        extension (str): File extension of the images to use
    
    Returns:
        bool: True if successful, False otherwise
    """
    # Check if folder exists
    if not os.path.exists(image_folder):
        print(f"Error: Directory '{image_folder}' not found.")
        return False
    
    # Find all image files in the directory
    image_files = glob.glob(os.path.join(image_folder, f"*.{extension}"))
    
    if not image_files:
        print(f"Error: No {extension} files found in '{image_folder}'")
        return False
    
    # Sort files naturally
    image_files.sort(key=natural_sort_key)
    print(f"Found {len(image_files)} images in '{image_folder}'")
    
    # Read first image to get dimensions
    first_image = cv2.imread(image_files[0])
    if first_image is None:
        print(f"Error: Could not read image '{image_files[0]}'")
        return False
    
    height, width, channels = first_image.shape
    
    # Define the codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Use 'XVID' for .avi format
    video_writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # Process each image
    print(f"Creating video: {output_path}")
    for image_file in tqdm(image_files):
        image = cv2.imread(image_file)
        if image is not None:
            video_writer.write(image)
    
    # Release the video writer
    video_writer.release()
    print(f"Video saved to {output_path}")
    
    return True

def create_stereo_videos(left_folder, right_folder, output_dir=None, fps=15, extension="jpg"):
    """
    Create separate videos from left and right image folders
    
    Args:
        left_folder (str): Path to folder containing left camera images
        right_folder (str): Path to folder containing right camera images
        output_dir (str): Directory to save output videos (default: current directory)
        fps (int): Frames per second for the output videos
        extension (str): File extension of the images to use
    """
    # Use current directory if output_dir is not specified
    if output_dir is None:
        output_dir = os.getcwd()
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Define output paths
    left_video_path = os.path.join(output_dir, "left_camera.mp4")
    right_video_path = os.path.join(output_dir, "right_camera.mp4")
    
    # Create videos
    left_success = create_video_from_images(left_folder, left_video_path, fps, extension)
    right_success = create_video_from_images(right_folder, right_video_path, fps, extension)
    
    # Check results
    if left_success and right_success:
        print("\nBoth videos created successfully:")
        print(f"Left camera video: {left_video_path}")
        print(f"Right camera video: {right_video_path}")
    else:
        print("\nThere were issues creating one or both videos.")

def interactive_stitch(left, right):
    script_dir = '/home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/'

    yaml_path = os.path.join(script_dir, 'rig_params.yaml')

    with open(yaml_path, 'r') as f:
        params = yaml.safe_load(f)

    dx = params['tx']
    dy = params['ty']
    scale = params['scale']
    theta = np.deg2rad(params['rot_deg'])

    affine_matrix = np.array([
            [scale * np.cos(theta), -scale * np.sin(theta), dx],
            [scale * np.sin(theta), scale * np.cos(theta), dy]
        ])
    h, w = left.shape[:2]

    corners = np.array([
        [0, 0],
        [0, h],
        [w, 0],
        [w, h]
    ], dtype=np.float32)

    transformed_corners = cv2.transform(np.array([corners]), affine_matrix)[0]
    all_corners = np.vstack((corners, transformed_corners))

    [xmin, ymin] = np.floor(all_corners.min(axis=0)).astype(int)
    [xmax, ymax] = np.ceil(all_corners.max(axis=0)).astype(int)
    output_size = (xmax - xmin, ymax - ymin)
    offset = np.array([-xmin, -ymin])

    affine_with_offset = affine_matrix.copy()
    affine_with_offset[:, 2] += offset

    transformed_right = cv2.warpAffine(right, affine_with_offset, output_size)

    canvas = np.zeros((output_size[1], output_size[0], 3), dtype=np.uint8)
    canvas[offset[1]:offset[1] + h, offset[0]:offset[0] + w] = left

    stitched = np.maximum(canvas, transformed_right)

    return stitched

def stitch_left_right_images(left_image_folder, right_image_folder, output_dir):
    """
    Stitch left and right images side by side
    
    Args:
        left_image_path (str): Path to the left image
        right_image_path (str): Path to the right image
        output_dir (str): Path to save the stitched image
    
    Returns:
        bool: True if successful, False otherwise
    """
    extension = "jpg"
    if not os.path.exists(left_image_folder):
        print(f"Error: Directory '{left_image_folder}' not found.")
        return False
    os.makedirs(output_dir, exist_ok=True)

    
    left_image_files = glob.glob(os.path.join(left_image_folder, "*.jpg"))
    right_image_files = glob.glob(os.path.join(right_image_folder, "*.jpg"))
    # Check if folders exist
    if not left_image_files:
        print(f"Error: No jpg files found in '{left_image_files}'")
        return False
    
    # Sort files naturally
    left_image_files.sort(key=natural_sort_key)
    right_image_files.sort(key=natural_sort_key)
    print(f"Found {len(left_image_files)} left images and {len(right_image_files)} right images")
    for left_img, right_img in zip(left_image_files, right_image_files):
        # print(f"Stitching {left_img} and {right_img}")
        # Read images
        left_image = cv2.imread(left_img)
        right_image = cv2.imread(right_img)
        
        if left_image is None or right_image is None:
            print("Error: Could not read one or both images.")
            return False
        
        stitched_image = interactive_stitch(left_image, right_image)
        
        # Save the stitched image
        output_path = os.path.join(output_dir, f"stitched_{os.path.basename(left_img)}")
        cv2.imwrite(output_path, stitched_image)
        print(f"Stitched image saved to {output_path}")

#### Example Use Case for One Folder

In [18]:
root = tk.Tk()
root.withdraw()  # Hide the main window

# folder_path = filedialog.askdirectory(
#     title="Select your video folder",
#     initialdir=os.path.expanduser("~") + "/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/video/"
# )
folder_path = '/home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/video/09_11_2025_15_29_15_tetherless_C'
if folder_path:  # Check if a file was selected
    # Convert to Path object for easier manipulation
    folder_path_obj = Path(folder_path)
    left_dir = folder_path_obj / "left"
    right_dir = folder_path_obj / "right"    
    print("Selected folder:", folder_path)
else:
    print("No folder selected")
    folder_path = ""
    npz_filename = ""

output_folder = folder_path + "/stitched/"
os.makedirs(output_folder, exist_ok=True)
stitch_left_right_images(left_dir, right_dir, output_folder)
print("Stitching completed.")

Selected folder: /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/video/09_11_2025_15_29_15_tetherless_C
Found 1337 left images and 1337 right images
Stitched image saved to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/video/09_11_2025_15_29_15_tetherless_C/stitched/stitched_frame0.jpg
Stitched image saved to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/video/09_11_2025_15_29_15_tetherless_C/stitched/stitched_frame1.jpg
Stitched image saved to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/video/09_11_2025_15_29_15_tetherless_C/stitched/stitched_frame2.jpg
Stitched image saved to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/video/09_11_2025_15_29_15_tetherless_C/stitched/stitched_frame3.jpg
Stitched image saved to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/vid

In [67]:
input_folder = '/home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/video/09_04_2025_NEA_video/04_16_2025_02_50_39/stitched'
create_video_from_images(input_folder, input_folder + "/stitched_video.mp4", fps=15, extension="jpg")

Found 193 images in '/home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/video/09_04_2025_NEA_video/04_16_2025_02_50_39/stitched'
Creating video: /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/video/09_04_2025_NEA_video/04_16_2025_02_50_39/stitched/stitched_video.mp4


100%|██████████| 193/193 [00:02<00:00, 80.23it/s]

Video saved to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/video/09_04_2025_NEA_video/04_16_2025_02_50_39/stitched/stitched_video.mp4


True

In [61]:
import numpy as np
import cv2 
import re
import glob
print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")
import json
import os
np.set_printoptions(precision=2, suppress=False)  # neat printing
from datetime import datetime

class StereoProcessor():
    def __init__(self):
        cv_params, __ = self.parse_cv_params()

        DIM=(640, 480)

        KL=np.array([[567.5108694496967, 0.0, 296.65113980779455], [0.0, 555.4066963178193, 209.8357339941157], [0.0, 0.0, 1.0]])
        DL=np.array([[2.3304306359821676], [-22.72098020337239], [96.18495273925535], [-142.38468386964402]])
        KR=np.array([[654.417602210813, 0.0, 236.58996237455327], [0.0, 625.1929731011552, 223.78494035869645], [0.0, 0.0, 1.0]])
        DR=np.array([[-1.4455817082295017], [22.451030896752517], [-138.62442550617396], [291.546248181601]])
        R=np.array([[0.9350531687261983, 0.001570069165988603, -0.3545040289445381], [-0.007541738734551825, 0.9998519807567543, -0.01546411179650275], [0.3544272758013351, 0.01713334350350178, 0.934926603915214]])
        T=np.array([[-2.1428084390335562], [-0.02825237240566071], [-0.36330855915337945]])*25.4

        KL=np.array([[708.3477312219868, 0.0, 260.69187590557686], [0.0, 675.3059166594338, 301.31936629865646], [0.0, 0.0, 1.0]])
        DL=np.array([[-0.39383047117877457], [6.721465255404687], [-35.99917141986595], [61.49579122578909]])
        KR=np.array([[667.0400978057647, 0.0, 334.8109094526051], [0.0, 644.922628956739, 364.07228200370565], [0.0, 0.0, 1.0]])
        DR=np.array([[0.8809516193294453], [-6.609640306922403], [21.549513701823056], [-24.149385093847197]])
        R=np.array([[0.8721459388442752, 0.02130940474841954, -0.4887815162490354], [-0.06589130347707366, 0.9950649291385254, -0.07418977641584823], [0.4847884048567273, 0.09691076342599622, 0.8692459412897253]])
        T=np.array([[-2.085136618149882], [0.1939622251215522], [-0.9258137973647751]])*25.4        ############# May 16th OG Intrinsics ##########################33

        R1,R2,P1,P2,self.Q = cv2.fisheye.stereoRectify(KL,DL,KR,DR,DIM,R,T, cv2.fisheye.CALIB_ZERO_DISPARITY)

        print(self.Q)
        self.L_undist_map=cv2.fisheye.initUndistortRectifyMap(KL,DL,np.identity(3),KL,DIM,cv2.CV_32FC1)
        self.R_undist_map=cv2.fisheye.initUndistortRectifyMap(KR,DR,np.identity(3),KR,DIM,cv2.CV_32FC1)
        self.left1, self.left2 = cv2.fisheye.initUndistortRectifyMap(KL, DL, R1, P1, DIM, cv2.CV_32FC1)
        self.right1, self.right2 = cv2.fisheye.initUndistortRectifyMap(KR,DR, R2, P2, DIM, cv2.CV_32FC1)
        self.stereo = cv2.StereoBM.create(numDisparities=cv_params["numDisparities"], blockSize=cv_params["blockSize"])
        self.stereo.setMinDisparity(cv_params["MinDisparity"])
        self.stereo.setTextureThreshold(cv_params["TextureThreshold"])

        #post filtering parameters: prevent false matches, help filter at boundaries
        self.stereo.setSpeckleRange(cv_params["SpeckleRange"])
        self.stereo.setSpeckleWindowSize(cv_params["SpeckleWindowSize"])
        self.stereo.setUniquenessRatio(cv_params["UniquenessRatio"])

        self.stereo.setDisp12MaxDiff(cv_params["Disp12MaxDiff"])
        self.depth_mean_list = [10000000]
        self.num_samples = 10
        t = datetime.today().strftime("%m_%d_%Y_%H_%M_%S")
        folder_name =  "video/" + t
        os.makedirs(folder_name + "/left")
        os.makedirs(folder_name + "/right")
        os.makedirs(folder_name + "/detection")
        os.makedirs(folder_name + "/depth")
        os.makedirs(folder_name + "/depth_data")
        self.depth_min = None
        self.depth_max = None
        self.output_folder = folder_name
        print(f"output path: {self.output_folder}")
        self.count = 0
        self.flag_turn = False
        self.TURN_RIGHT = None


    def stereo_params_update(self):
        if self._last_update != os.stat('cv_config.json').st_mtime:
            cv_params, __ = self.parse_cv_params()
            self.stereo.setNumDisparities(cv_params["numDisparities"])
            self.stereo.setBlockSize(cv_params["blockSize"])
            self.stereo.setMinDisparity(cv_params["MinDisparity"])
            self.stereo.setTextureThreshold(cv_params["TextureThreshold"])

            #post filtering parameters: prevent false matches, help filter at boundaries
            self.stereo.setSpeckleRange(cv_params["SpeckleRange"])
            self.stereo.setSpeckleWindowSize(cv_params["SpeckleWindowSize"])
            self.stereo.setUniquenessRatio(cv_params["UniquenessRatio"])

            self.stereo.setDisp12MaxDiff(cv_params["Disp12MaxDiff"])
    def parse_cv_params(self):
        with open('../cv_config.json') as config:
            param = json.load(config)
            self._last_update = os.fstat(config.fileno()).st_mtime
        print(f"[MESSAGE] Config: {param}\n")    
        # Serializing json
        config_params = json.dumps(param, indent=14)
        return param, config_params
    def update(self, left, right):
        print(f"count: {self.count}")
        x_planning_bounds = [157, 425]
        cv2.imwrite(self.output_folder + "/left/frame%d.jpg" % self.count, left)
        cv2.imwrite(self.output_folder + "/right/frame%d.jpg" % self.count, right)

        fixedLeft = cv2.remap(left, self.left1, self.left2, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)
        fixedRight = cv2.remap(right, self.right1, self.right2, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)

        grayLeft = cv2.cvtColor(fixedLeft, cv2.COLOR_BGR2GRAY)
        grayRight = cv2.cvtColor(fixedRight, cv2.COLOR_BGR2GRAY)
        disparity = self.stereo.compute(grayLeft,grayRight)
        denoise = 5
        noise=cv2.erode(disparity,np.ones((denoise,denoise)))
        noise=cv2.dilate(noise,np.ones((denoise,denoise)))
        disparity = cv2.medianBlur(noise, ksize=5)
        # norm_disparity = cv2.normalize(disparity, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)
        invalid_pixels = disparity < 0.0001
        disparity[invalid_pixels] = 0
        norm_disparity = np.array((disparity/16.0 - self.stereo.getMinDisparity())/self.stereo.getNumDisparities(), dtype='f')
        self.norm_disparity = norm_disparity
        points3D = cv2.reprojectImageTo3D(np.array(disparity/16.0/1000.0, dtype='f'), self.Q, handleMissingValues=True)
        depth = self.Q[2,3]/self.Q[3,2]/np.array(disparity/16.0, dtype='f')/1000
        x_bounds = [158,613]
        y_bounds = [30,450]
        depth_window = depth[230:250,358:413]

        finite_depth = depth_window[np.isfinite(depth_window)]
        stop_mean = np.median(finite_depth)
        h_thresh = 80
        w_thresh = 100
        depth[np.isinf(depth)] = np.median(finite_depth)
        # print(np.mean(depth))
        
        depth_thresh = 1.5 # Threshold for SAFE distance (in cm)
        contour_vis = cv2.applyColorMap(
            cv2.convertScaleAbs(
                np.clip(depth, 0, 5) * 51, 
                alpha=1
            ), 
            cv2.COLORMAP_PLASMA
        )

        # Mask to segment regions with depth less than threshold
        mask = cv2.inRange(depth,0.1,depth_thresh)

        print(f"Depth mean list: {self.depth_mean_list}")
        # Check if a significantly large obstacle is present and filter out smaller noisy regions
        x_bounds = [157, 425]
        y_bounds = [30, 450]

        # Extract coordinates
        x_min, x_max = x_bounds
        y_min, y_max = y_bounds

        # Draw rectangle using the bounds
        # cv2.rectangle(contour_vis, (x_min, y_min), (x_max, y_max), (0, 0, 255), 3)
        if np.sum(mask)/255.0 > 0.01*mask.shape[0]*mask.shape[1]:
            print("drawing contour")
            # Contour detection 
            contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
            cnts = sorted(contours, key=cv2.contourArea, reverse=True)
            print(f"Mask sum: {np.sum(mask)/255.0}, threshold: {0.01*mask.shape[0]*mask.shape[1]}")

            # Check if detected contour is significantly large (to avoid multiple tiny regions)
            cv2.drawContours(contour_vis, cnts[0], -1, (0, 255, 0), 2)
            if cv2.contourArea(cnts[0]) > 0.03*mask.shape[0]*mask.shape[1]:
                x,y,w,h = cv2.boundingRect(cnts[0])
                x_center = int(x + w/2)
                y_center = int(y + h/2)
                # cv2.rectangle(contour_vis, (x, y), (x+w, y+h), (0, 0, 255), 3)
                # finding average depth of region represented by the largest contour 
                mask2 = np.zeros_like(mask)
                # Calculating the average depth of the object closer than the safe distance
                depth_mean, _ = cv2.meanStdDev(depth, mask=mask)
                
                depth_mean, _ = cv2.meanStdDev(depth, mask=mask2)
                if not np.isnan(depth_mean).any() and depth_mean < 15:
                    if len(self.depth_mean_list) < self.num_samples:
                        self.depth_mean_list.append(depth_mean[0,0])
                    else:
                        self.depth_mean_list.pop(0)
                        self.depth_mean_list.append(depth_mean[0,0])
                    
                # print(depth_mean[0])
                filtered_depth = np.mean(self.depth_mean_list)
                # return filtered_depth, x_center, y_center, norm_disparity
            else:
                depth_mean = np.mean(depth)
                if not np.isnan(depth_mean).any() and depth_mean < 15:
                    if len(self.depth_mean_list) < self.num_samples:
                        self.depth_mean_list.append(np.mean(depth))
                    else:
                        self.depth_mean_list.pop(0)
                        self.depth_mean_list.append(np.mean(depth))
                filtered_depth = np.mean(self.depth_mean_list)
                # return filtered_depth, None, None, norm_disparity
                x_center, y_center = 0, 0
        else:
            depth_mean = np.mean(depth)
            if not np.isnan(depth_mean).any() and depth_mean < 15:
                if len(self.depth_mean_list) < self.num_samples:
                        self.depth_mean_list.append(np.mean(depth))
                else:
                    self.depth_mean_list.pop(0)
                    self.depth_mean_list.append(np.mean(depth))
            filtered_depth = np.mean(self.depth_mean_list)
            x_center, y_center = 0, 0

        mid_x = (x_planning_bounds[0] + x_planning_bounds[1]) / 2
        print(f"Filtered depth: {filtered_depth}, x_center: {x_center}, y_center: {y_center}")

        if self.flag_turn:
            if filtered_depth > 3.0 and self.flag_turn:
                self.flag_turn = False
                self.TURN_RIGHT = None

        else:

            if filtered_depth < 3.5 and not self.flag_turn:
                # cv2.line(contour_vis, 
                #         (int(mid_x), y_bounds[0]), 
                #         (int(mid_x), y_bounds[1]), 
                #         (255, 255, 0), 2)
                # cv2.circle(contour_vis, (x_center, y_center), 10, (255, 255, 0), -1)

                
                # Indicate turn direction
                if (x_center - 157) / (425 - 157) > 0.5:
                    self.TURN_RIGHT = False
                else:
                    self.TURN_RIGHT = True

                turn_text = "TURN RIGHT" if self.TURN_RIGHT else "TURN LEFT"
                cv2.putText(contour_vis, turn_text, 
                            (int(mid_x) - 280, y_bounds[0]),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)

            if filtered_depth < 3.0 and not self.flag_turn:
                self.flag_turn = True
                # Highlight decision boundaries for turning
                mid_x = (x_planning_bounds[0] + x_planning_bounds[1]) / 2
                cv2.line(contour_vis, 
                        (int(mid_x), y_bounds[0]), 
                        (int(mid_x), y_bounds[1]), 
                        (255, 255, 0), 2)
                # cv2.circle(contour_vis, (x_center, y_center), 10, (255, 255, 0), -1)
                # Indicate turn direction
                if (x_center - 157) / (425 - 157) > 0.5:
                    self.TURN_RIGHT = False
                else:
                    self.TURN_RIGHT = True

                turn_text = "TURN RIGHT" if self.TURN_RIGHT else "TURN LEFT"
                cv2.putText(contour_vis, turn_text, 
                            (int(mid_x) - 280, y_bounds[0]),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)
            
                
        print(f"count: {self.count}")
        if self.count == 0:
            first_depth = norm_disparity
            # np.savez_compressed(os.path.join(depth_folder, f"depth_{count:04d}.npz"), depth=depth)
            self.depth_min = np.min(first_depth[~np.isnan(first_depth) & ~np.isinf(first_depth)])
            self.depth_max = np.max(first_depth[~np.isnan(first_depth) & ~np.isinf(first_depth)])
            first_depth = np.nan_to_num(first_depth, nan=self.depth_min, posinf=self.depth_max, neginf=self.depth_min)
            normalized_depth = ((first_depth - self.depth_min) / (self.depth_max - self.depth_min) * 255).astype(np.uint8)
        normalized_depth = ((norm_disparity - self.depth_min) / (self.depth_max - self.depth_min) * 255).astype(np.uint8)
        depth_color = cv2.applyColorMap(normalized_depth, cv2.COLORMAP_VIRIDIS)
        np.save(self.output_folder + "/depth_data/frame%d.npy" % self.count, [filtered_depth, x_center, y_center])
        cv2.imwrite(self.output_folder + "/depth/frame%d.jpg" % self.count, depth_color)
        cv2.imwrite(self.output_folder + "/detection/frame%d.jpg" % self.count, contour_vis)

        self.count += 1
        return filtered_depth, x_center, y_center, norm_disparity



def natural_sort_key(s):
    """
    Sort strings containing numbers naturally (e.g., frame1.npy, frame2.npy, ..., frame10.npy)
    """
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]


OpenCV version: 4.6.0
NumPy version: 1.26.4


In [62]:
sp = StereoProcessor()
image_dir = '/home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/video/09_04_2025_NEA_video/04_16_2025_03_13_52_tetherless_C/'
left_folder = os.path.join(image_dir, 'left')
right_folder = os.path.join(image_dir, 'right')
extension = 'jpg'
left_files = glob.glob(os.path.join(left_folder, f"*.{extension}"))
right_files = glob.glob(os.path.join(right_folder, f"*.{extension}"))
left_files.sort(key=natural_sort_key)
right_files.sort(key=natural_sort_key)

# Ensure all folders have the same number of files
num_frames = min(len(left_files), len(right_files))
print(f"Using {num_frames} frames from each folder")

left_files = left_files[:num_frames]
right_files = right_files[:num_frames]


for count in range(num_frames):
    left = cv2.imread(left_files[count])
    right = cv2.imread(right_files[count])
    depth, x, y, disp = sp.update(left, right)
    print(f"Depth: {depth}, x: {x}, y: {y}")

[MESSAGE] Config: {'phi': 0.0, 'minThreshold': 10, 'maxThreshold': 200, 'minArea': 50, 'maxArea': 100000, 'minCircularity': 0.1, 'minConvexity': 0.1, 'minInertiaRatio': 0.1, 'lower_yellow': [10, 35, 50], 'upper_yellow': [30, 70, 100], 'turn_thresh': 150, 'dive_thresh': 100, 'num_cycles': 1, 'numDisparities': 128, 'blockSize': 61, 'MinDisparity': 0, 'TextureThreshold': 0, 'SpeckleRange': 2, 'SpeckleWindowSize': 5, 'UniquenessRatio': 2, 'Disp12MaxDiff': 2}

[[ 1.00e+00  0.00e+00  0.00e+00 -1.68e+02]
 [ 0.00e+00  1.00e+00  0.00e+00 -3.31e+02]
 [ 0.00e+00  0.00e+00  0.00e+00  5.73e+02]
 [ 0.00e+00  0.00e+00  1.72e-02 -0.00e+00]]
output path: video/09_11_2025_15_29_15
Using 1450 frames from each folder
count: 0
Depth mean list: [10000000]
drawing contour
Mask sum: 304456.0, threshold: 3072.0
Filtered depth: 5000000.0, x_center: 320, y_center: 240
count: 0
Depth: 5000000.0, x: 320, y: 240
count: 1


/tmp/ipykernel_11810/3594893051.py:111: RuntimeWarning: divide by zero encountered in divide
  depth = self.Q[2,3]/self.Q[3,2]/np.array(disparity/16.0, dtype='f')/1000
/usr/lib/python3/dist-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/lib/python3/dist-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)


Depth mean list: [10000000, 0.0]
drawing contour
Mask sum: 4933.0, threshold: 3072.0
Filtered depth: 5000000.0, x_center: 0, y_center: 0
count: 1
Depth: 5000000.0, x: 0, y: 0
count: 2
Depth mean list: [10000000, 0.0]
drawing contour
Mask sum: 5962.0, threshold: 3072.0
Filtered depth: 5000000.0, x_center: 0, y_center: 0
count: 2
Depth: 5000000.0, x: 0, y: 0
count: 3
Depth mean list: [10000000, 0.0]
drawing contour
Mask sum: 5948.0, threshold: 3072.0
Filtered depth: 5000000.0, x_center: 0, y_center: 0
count: 3
Depth: 5000000.0, x: 0, y: 0
count: 4
Depth mean list: [10000000, 0.0]
drawing contour
Mask sum: 5945.0, threshold: 3072.0
Filtered depth: 5000000.0, x_center: 0, y_center: 0
count: 4
Depth: 5000000.0, x: 0, y: 0
count: 5
Depth mean list: [10000000, 0.0]
drawing contour
Mask sum: 7458.0, threshold: 3072.0
Filtered depth: 5000000.0, x_center: 0, y_center: 0
count: 5
Depth: 5000000.0, x: 0, y: 0
count: 6
Depth mean list: [10000000, 0.0]
drawing contour
Mask sum: 9676.0, threshold: 30

error: OpenCV(4.6.0) ./modules/imgcodecs/src/loadsave.cpp:801: error: (-215:Assertion failed) !_img.empty() in function 'imwrite'


### Video Generation plus StereoProcessing

In [75]:
image_dir = '/home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/video/09_11_2025_14_48_26_tetherless_A/'
output_path = image_dir + "obstacle_avoidance_video.mp4"
fps = 3
save_depth = True
flag_turn = False
try:
    depth_folder = os.path.join(image_dir, "depth_data")
    os.makedirs(depth_folder, exist_ok=False)
    save_depth = True
except:
    save_depth = False
first_round = True
left_folder = image_dir + "left"
right_folder = image_dir + "right"
depth_folder = image_dir + "depth"
obstacle_folder = image_dir + "detection"
print(left_folder, right_folder, depth_folder, obstacle_folder)
depth_data = image_dir + "depth_data"
extension = 'jpg'
left_files = glob.glob(os.path.join(left_folder, f"*.{extension}"))
right_files = glob.glob(os.path.join(right_folder, f"*.{extension}"))
depth_files = glob.glob(os.path.join(depth_folder, f"*.{extension}"))
obstacle_files = glob.glob(os.path.join(obstacle_folder, f"*.{extension}"))
depth_data_files = glob.glob(os.path.join(depth_data, f"*.npy"))

left_files.sort(key=natural_sort_key)
right_files.sort(key=natural_sort_key)
depth_files.sort(key=natural_sort_key)
obstacle_files.sort(key=natural_sort_key)   
depth_data_files.sort(key=natural_sort_key)
num_frames = min(len(left_files), len(right_files))
print(f"Using {num_frames} frames from each folder")

left_files = left_files[:num_frames]
right_files = right_files[:num_frames]

# Read first images to get dimensions
first_left = cv2.imread(left_files[0])
first_right = cv2.imread(right_files[0])
first_depth = cv2.imread(depth_files[0])
TURN_RIGHT = None
if first_left is None or first_right is None:
    raise Exception("Could not read one or both of the first images")
else:
    h_left, w_left, _ = first_left.shape
    h_right, w_right, _ = first_right.shape
    print(first_depth.shape)
    h_depth, w_depth = first_depth.shape[0], first_depth.shape[1]

    # Resize all images to the same height
    target_height = min(h_left, h_right, h_depth)

    # Calculate new widths maintaining aspect ratio
    w_left_new = int(w_left * target_height / h_left)
    w_right_new = int(w_right * target_height / h_right)
    w_depth_new = int(w_depth * target_height / h_depth)

    # Compute composite dimensions
    # We'll place images in a row: [Left | Right | Depth | Obstacle]
    composite_width = w_left_new + w_right_new + w_depth_new + w_depth_new
    composite_height = target_height + 30  # Extra space for frame counter at the bottom

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(output_path, fourcc, fps, (composite_width, composite_height))


    # Process each set of images
    print(f"Creating obstacle avoidance video: {output_path}")


for count in range(len(left_files)):
    print(f"frame: {count}")
    filtered_depth = depth_data_files[count]
    filtered_depth, x_center, y_center = np.load(filtered_depth)
    print(filtered_depth)
    left = cv2.imread(left_files[count])
    right = cv2.imread(right_files[count])
    depth = cv2.imread(depth_files[count])
    obstacle = cv2.imread(obstacle_files[count])
    # Resize images to target height
    left_resized = cv2.resize(left, (w_left_new, target_height))
    right_resized = cv2.resize(right, (w_right_new, target_height))
    depth_resized = cv2.resize(depth, (w_depth_new, target_height))
    obstacle_resized = cv2.resize(obstacle, (w_depth_new, target_height))

    # Create composite frame with black background
    composite = np.zeros((composite_height, composite_width, 3), dtype=np.uint8)

    # Place images side by side
    composite[:target_height, :w_left_new] = left_resized
    composite[:target_height, w_left_new:w_left_new+w_right_new] = right_resized
    composite[:target_height, w_left_new+w_right_new:w_left_new+w_right_new+w_depth_new] = depth_resized
    composite[:target_height, w_left_new+w_right_new+w_depth_new:] = obstacle_resized

    # Add labels
    cv2.putText(composite, "Left", (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    cv2.putText(composite, "Right", (w_left_new + 10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    cv2.putText(composite, "Depth", (w_left_new + w_right_new + 10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    cv2.putText(composite, "Obstacle Detection", (w_left_new + w_right_new + w_depth_new + 10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    # Add frame counter
    frame_text = f"Frame: {count} DEPTH: {filtered_depth:.2f}m X: {x_center} Y: {y_center}"
    text_size = cv2.getTextSize(frame_text, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)[0]
    text_x = (composite_width - text_size[0]) // 2
    cv2.putText(composite, frame_text, (text_x, composite_height - 10), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    # Add to video
    video_writer.write(composite)

# Release video writer
video_writer.release()
print(f"video saved to {output_path}")
print(fps)

# print(f"stereo depth: {stereo_depth_data[0:30]}")


/home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/video/09_11_2025_14_48_26_tetherless_A/left /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/video/09_11_2025_14_48_26_tetherless_A/right /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/video/09_11_2025_14_48_26_tetherless_A/depth /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/video/09_11_2025_14_48_26_tetherless_A/detection
Using 527 frames from each folder
(480, 640, 3)
Creating obstacle avoidance video: /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/video/09_11_2025_14_48_26_tetherless_A/obstacle_avoidance_video.mp4
frame: 0
10000000.0
frame: 1
5000000.0
frame: 2
3333333.3333333335
frame: 3
2500000.0
frame: 4
2500000.0
frame: 5
2500000.0
frame: 6
2000000.4495256424
frame: 7
1666667.4091621637
frame: 8
1666667.4091621637
frame: 9
1428572.0649961403
frame: 10


In [ ]:
image_dir = '/home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/video/09_11_2025_14_48_26_tetherless_A/'
left_folder = image_dir + "left"
right_folder = image_dir + "right"


stitch_left_right_images(left_folder, right_folder, image_dir + "stitched")

Found 1337 left images and 1337 right images
Stitched image saved to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/video/09_11_2025_15_29_15_tetherless_C/stitched/stitched_frame0.jpg
Stitched image saved to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/video/09_11_2025_15_29_15_tetherless_C/stitched/stitched_frame1.jpg
Stitched image saved to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/video/09_11_2025_15_29_15_tetherless_C/stitched/stitched_frame2.jpg
Stitched image saved to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/video/09_11_2025_15_29_15_tetherless_C/stitched/stitched_frame3.jpg
Stitched image saved to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/video/09_11_2025_15_29_15_tetherless_C/stitched/stitched_frame4.jpg
Stitched image saved to /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware

### Use Stored Data Video Generation

In [15]:
import cv2
import numpy as np
import os
import glob

class VariableFPSVideoWriter:
    """
    Video writer that preserves actual variable frame rates
    """
    def __init__(self, output_path, dimensions):
        self.output_path = output_path
        self.dimensions = dimensions
        self.fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        self.frames_data = []  # Store (frame, fps) pairs
        
    def add_frame_with_fps(self, frame, measured_fps, frame_index):
        """Add frame with its actual measured FPS"""
        # Clamp FPS to reasonable video bounds
        actual_fps = max(0.5, min(measured_fps, 60.0))
        self.frames_data.append((frame.copy(), actual_fps))
        
    def create_variable_fps_video(self):
        """
        Create video where each frame duration matches the actual capture timing
        """
        if not self.frames_data:
            print("No frames to write!")
            return
            
        print(f"Creating variable FPS video with {len(self.frames_data)} frames...")
        
        # Calculate statistics
        fps_values = [fps for _, fps in self.frames_data]
        print(f"FPS range: {min(fps_values):.2f} - {max(fps_values):.2f}")
        print(f"Average FPS: {np.mean(fps_values):.2f}")
        
        # Use the median FPS as the base video FPS for encoding
        # This gives us good quality while allowing variable timing
        base_fps = np.median(fps_values)
        print(f"Using base encoding FPS: {base_fps:.2f}")
        
        # Create video writer with base FPS
        writer = cv2.VideoWriter(self.output_path, self.fourcc, base_fps, self.dimensions)
        
        total_frames_written = 0
        total_time = 0.0
        
        for i, (frame, actual_fps) in enumerate(self.frames_data):
            # Calculate how long this frame should be displayed
            frame_duration = 1.0 / actual_fps
            total_time += frame_duration
            
            # Calculate how many times to repeat this frame at base_fps
            # to achieve the correct duration
            repeat_count = max(1, round(frame_duration * base_fps))
            
            # Write frame the calculated number of times
            for _ in range(repeat_count):
                writer.write(frame)
                total_frames_written += 1
            
            if i % 50 == 0:
                print(f"Processed {i}/{len(self.frames_data)} frames")
        
        writer.release()
        
        print(f"✅ Variable FPS video created!")
        print(f"📁 Output: {self.output_path}")
        print(f"⏱️  Total capture time: {total_time:.2f} seconds")
        print(f"🎬 Video frames written: {total_frames_written}")
        print(f"📊 Video plays back the actual robot timing!")
        
    def release(self):
        """Placeholder for compatibility"""
        pass

# Your existing code with Method 1 integration
def main():
    image_dir = '/home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/video/09_11_2025_15_03_38_tetherless_B/'
    output_path = image_dir + "obstacle_avoidance_video_variable_frame_rate.mp4"
    fps_data = np.load('fps_data.npy')

    # Video creation settings
    # No target_fps needed - we'll use actual variable FPS!

    save_depth = False
    flag_turn = False
    try:
        depth_folder = os.path.join(image_dir, "depth_data")
        os.makedirs(depth_folder, exist_ok=False)
        save_depth = True
    except:
        save_depth = False
    first_round = True
    left_folder = image_dir + "left"
    right_folder = image_dir + "right"
    depth_folder = image_dir + "depth"
    obstacle_folder = image_dir + "detection"
    print(left_folder, right_folder, depth_folder, obstacle_folder)
    depth_data = image_dir + "depth_data"
    stitched_folder = image_dir + "stitched"
    combined_folder = image_dir + "combined"
    os.makedirs(combined_folder, exist_ok=True)
    extension = 'jpg'
    left_files = glob.glob(os.path.join(left_folder, f"*.{extension}"))
    right_files = glob.glob(os.path.join(right_folder, f"*.{extension}"))
    depth_files = glob.glob(os.path.join(depth_folder, f"*.{extension}"))
    obstacle_files = glob.glob(os.path.join(obstacle_folder, f"*.{extension}"))
    combined_files = glob.glob(os.path.join(stitched_folder, f"*.{extension}"))

    depth_data_files = glob.glob(os.path.join(depth_data, f"*.npy"))

    left_files.sort(key=natural_sort_key)
    right_files.sort(key=natural_sort_key)
    depth_files.sort(key=natural_sort_key)
    obstacle_files.sort(key=natural_sort_key)   
    combined_files.sort(key=natural_sort_key)
    depth_data_files.sort(key=natural_sort_key)
    num_frames = min(len(left_files), len(right_files))
    print(f"Using {num_frames} frames from each folder")

    left_files = left_files[:num_frames]
    right_files = right_files[:num_frames]

    # Check if stitched frames already exist
    stitched_files = glob.glob(os.path.join(stitched_folder, f"*.{extension}"))
    stitched_files.sort(key=natural_sort_key)
    
    print("No stitched frames found, will create composite frames")
    # Read first images to get dimensions for creating composites
    first_left = cv2.imread(left_files[0])
    first_right = cv2.imread(right_files[0])
    first_depth = cv2.imread(depth_files[0])
    first_combined = cv2.imread(stitched_files[0])
    
    if first_left is None or first_right is None:
        raise Exception("Could not read one or both of the first images")
    
    h_left, w_left, _ = first_left.shape
    h_right, w_right, _ = first_right.shape
    h_combined, w_combined, _ = first_combined.shape
    print(first_depth.shape)
    h_depth, w_depth = first_depth.shape[0], first_depth.shape[1]

    # Resize all images to the same height
    target_height = min(h_combined, h_depth)

    # Calculate new widths maintaining aspect ratio
    w_left_new = int(w_left * target_height / h_left)
    w_right_new = int(w_right * target_height / h_right)
    w_depth_new = int(w_depth * target_height / h_depth)
    w_combined_new = int(w_combined * target_height / h_combined)

    # Compute composite dimensions
    # We'll place images in a row: [Combined | Depth | Obstacle]
    composite_width = w_combined_new + w_depth_new + w_depth_new
    composite_height = target_height + 30  # Extra space for frame counter at the bottom
    use_existing_stitched = False

    # Create variable FPS video writer
    video_writer = VariableFPSVideoWriter(output_path, (composite_width, composite_height))

    # Process each set of images
    print(f"Creating variable FPS obstacle avoidance video: {output_path}")
    print("Video will preserve actual robot timing - slow periods will play slowly, fast periods quickly!")

    # Main processing loop
    frames_to_process = min(len(left_files), len(stitched_files) if use_existing_stitched else len(left_files))
    
    for count in range(frames_to_process):
        print(f"Processing frame: {count}")
        
        # Get real-time fps for this frame
        current_fps = fps_data[count] if count < len(fps_data) else 3  # fallback
        # Clamp FPS to reasonable bounds
        current_fps = max(0.1, min(current_fps, 60.0))
        
        # Create composite frame (your original logic)
        filtered_depth = depth_data_files[count]
        filtered_depth, x_center, y_center = np.load(filtered_depth)
        print(f"FPS: {current_fps:.2f}, Depth: {filtered_depth:.2f}")
        
        left = cv2.imread(left_files[count])
        right = cv2.imread(right_files[count])
        combined = cv2.imread(stitched_files[count])
        depth = cv2.imread(depth_files[count])
        obstacle = cv2.imread(obstacle_files[count])
        
        # Resize images to target height
        left_resized = cv2.resize(left, (w_left_new, target_height))
        right_resized = cv2.resize(right, (w_right_new, target_height))
        depth_resized = cv2.resize(depth, (w_depth_new, target_height))
        combined_resized = cv2.resize(combined, (w_combined_new, target_height))
        obstacle_resized = cv2.resize(obstacle, (w_depth_new, target_height))

        # Create composite frame with black background
        composite = np.zeros((composite_height, composite_width, 3), dtype=np.uint8)

        # Place images side by side
        composite[:target_height, :w_combined_new] = combined_resized
        composite[:target_height, w_combined_new:w_combined_new+w_depth_new] = depth_resized
        composite[:target_height, w_combined_new+w_depth_new:w_combined_new + w_depth_new + w_depth_new] = obstacle_resized

        # Add labels
        cv2.putText(composite, "Turtle POV", (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        cv2.putText(composite, "Depth", (w_combined_new + 10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        cv2.putText(composite, "Obstacle Detection", (w_combined_new+w_depth_new + 10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        # Add frame counter with FPS info
        frame_text = f"Frame: {count} DEPTH: {filtered_depth:.2f}m X: {x_center} Y: {y_center}"
        text_size = cv2.getTextSize(frame_text, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)[0]
        text_x = (composite_width - text_size[0]) // 2
        cv2.putText(composite, frame_text, (text_x, composite_height - 10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        
        combined_filename = f"combined_{count:04d}.jpg"  # e.g., combined_0001.jpg
        combined_path = os.path.join(combined_folder, combined_filename)

        cv2.imwrite(combined_path, composite)
        
        # Add frame to variable FPS writer
        # video_writer.add_frame_with_fps(composite, current_fps, count)

    # Create the actual variable FPS video
    # video_writer.create_variable_fps_video()
    # video_writer.release()

    print(f"\n✅ Variable FPS video creation complete!")
    print(f"📁 Output: {output_path}")
    print(f"🤖 Video shows actual robot performance timing!")
    print(f"⚡ Fast periods = quick playback, slow periods = slow playback")

if __name__ == "__main__":
    main()

/home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/video/09_11_2025_15_03_38_tetherless_B/left /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/video/09_11_2025_15_03_38_tetherless_B/right /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/video/09_11_2025_15_03_38_tetherless_B/depth /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/video/09_11_2025_15_03_38_tetherless_B/detection
Using 335 frames from each folder
No stitched frames found, will create composite frames
(480, 640, 3)
Creating variable FPS obstacle avoidance video: /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/turtle_analysis/video/09_11_2025_15_03_38_tetherless_B/obstacle_avoidance_video_variable_frame_rate.mp4
Video will preserve actual robot timing - slow periods will play slowly, fast periods quickly!
Processing frame: 0
FPS: 0.34, Depth: 10000000.00
Processing fr

In [ ]:
import os, glob

# Suppose each file is named like: frame_1234567890.123.png (unix timestamp)
files = sorted(glob.glob("frame_*.png"))
with open("timestamps.txt", "w") as f:
    for i in range(len(files)-1):
        t0 = float(files[i].split("_")[1].split(".")[0] + "." + files[i].split("_")[1].split(".")[1])
        t1 = float(files[i+1].split("_")[1].split(".")[0] + "." + files[i+1].split("_")[1].split(".")[1])
        dt = t1 - t0
        f.write(f"file '{files[i]}'\n")
        f.write(f"duration {dt}\n")
    f.write(f"file '{files[-1]}'\n")  # last frame

In [18]:
import os
import glob
import time
from pathlib import Path
import numpy as np
from datetime import datetime
import csv


def natural_sort_key(s):
    """Natural sorting for filenames with numbers"""
    import re
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', s)]


def read_timestamp_file(timestamp_file):
    """
    Read timestamps from CSV file
    
    Args:
        timestamp_file: Path to timestamp CSV file
    
    Returns:
        list: List of (frame_number, unix_timestamp, human_readable) tuples
    """
    timestamps = []
    
    try:
        with open(timestamp_file, 'r') as f:
            reader = csv.reader(f)
            for line_num, row in enumerate(reader):
                # Skip header and empty lines
                if line_num == 0 or not row or row[0].startswith('#'):
                    continue
                
                if len(row) >= 3:
                    frame_num = int(row[0])
                    unix_timestamp = float(row[1])
                    human_readable = row[2]
                    timestamps.append((frame_num, unix_timestamp, human_readable))
                    
    except Exception as e:
        print(f"Error reading timestamp file {timestamp_file}: {e}")
        return []
    
    return timestamps


def calculate_frame_durations_from_file(timestamps):
    """
    Calculate frame durations from timestamp file data
    
    Args:
        timestamps: List of (frame_number, unix_timestamp, human_readable) tuples
    
    Returns:
        list: Frame durations in seconds
    """
    
    if len(timestamps) < 2:
        print("Need at least 2 frames to calculate durations")
        return []
    
    durations = []
    print("\nFrame Durations from timestamp file:")
    
    for i in range(len(timestamps) - 1):
        # Time difference between consecutive frames
        dt = timestamps[i+1][1] - timestamps[i][1]  # unix_timestamp difference
        durations.append(dt)
        
        frame_curr = timestamps[i][0]
        frame_next = timestamps[i+1][0]
        print(f"Frame {frame_curr} → {frame_next}: {dt:.3f} seconds")
    
    # Add duration for last frame (use average of previous durations)
    if durations:
        avg_duration = np.mean(durations)
        durations.append(avg_duration)
        print(f"Last frame duration (estimated): {avg_duration:.3f} seconds")
    
    return durations


def create_ffmpeg_concat_from_timestamp_file(image_dir, timestamp_file, target_folder="combined"):
    """
    Create FFmpeg concat file using timestamps from file
    
    Args:
        image_dir: Base directory containing images
        timestamp_file: Path to timestamp CSV file
        target_folder: Folder with processed frames to use in video
    """
    
    target_dir = os.path.join(image_dir, target_folder)
    
    print(f"Reading timestamps from: {timestamp_file}")
    print(f"Using frames from: {target_folder}")
    
    # Read timestamps from file
    timestamps = read_timestamp_file(timestamp_file)
    
    if not timestamps:
        print(f"No timestamps found in {timestamp_file}")
        return None
    
    # Get processed files (for video content)
    target_files = glob.glob(os.path.join(target_dir, "*.jpg"))
    target_files.sort(key=natural_sort_key)
    
    if not target_files:
        print(f"No .jpg files found in {target_dir}")
        return None
    
    print(f"\nAnalyzing timestamps from file: {len(timestamps)} entries...")
    
    # Print first few for verification
    print("First few timestamp entries:")
    for i, (frame_num, unix_time, human_time) in enumerate(timestamps[:5]):
        print(f"  Frame {frame_num}: {human_time} (Unix: {unix_time:.6f})")
    
    print(f"Capture Analysis:")
    first_time = timestamps[0][1]
    last_time = timestamps[-1][1]
    total_time = last_time - first_time
    
    print(f"   First capture: {timestamps[0][2]}")
    print(f"   Last capture:  {timestamps[-1][2]}")
    print(f"   Total experiment time: {total_time:.2f} seconds")
    print(f"   Average capture rate: {len(timestamps) / total_time:.2f} FPS")
    
    # Calculate frame durations from timestamps
    durations = calculate_frame_durations_from_file(timestamps)
    
    # Create FFmpeg concat file
    timestamp_filename = os.path.basename(timestamp_file).replace('.txt', '')
    concat_file = os.path.join(image_dir, f"concat_from_{timestamp_filename}.txt")
    
    frames_written = 0
    with open(concat_file, "w") as f:
        # Use minimum of timestamps and target frames
        num_frames = min(len(timestamps), len(target_files))
        
        for i in range(num_frames):
            # Match by frame number if possible, otherwise by index
            if i < len(target_files):
                target_filename = os.path.basename(target_files[i])
                f.write(f"file '{target_folder}/{target_filename}'\n")
                
                if i < len(durations):
                    # Clamp duration to reasonable video bounds
                    duration = max(0.001, min(durations[i], 30.0))  # 1ms to 30s per frame
                    f.write(f"duration {duration:.6f}\n")
                    frames_written += 1
        
        # Add last frame (FFmpeg requirement)
        if num_frames > 0:
            last_target = os.path.basename(target_files[min(num_frames-1, len(target_files)-1)])
            f.write(f"file '{target_folder}/{last_target}'\n")
    
    print(f"\nCreated FFmpeg concat file: {concat_file}")
    print(f"Frames: {frames_written} with timing from timestamp file")
    
    # Show timing statistics
    valid_durations = [d for d in durations if 0.001 <= d <= 30.0]
    if valid_durations:
        print(f"Frame timing statistics:")
        print(f"   Fastest frame: {min(valid_durations):.3f}s ({1/min(valid_durations):.1f} FPS)")
        print(f"   Slowest frame: {max(valid_durations):.3f}s ({1/max(valid_durations):.1f} FPS)")
        print(f"   Average: {np.mean(valid_durations):.3f}s ({1/np.mean(valid_durations):.1f} FPS)")
        print(f"   Median: {np.median(valid_durations):.3f}s ({1/np.median(valid_durations):.1f} FPS)")
    
    return concat_file


def analyze_left_timestamps(left_timestamp_file):
    """
    Analyze left camera timestamp data for quality and consistency
    """
    print("\nAnalyzing left camera timestamp data...")
    
    left_timestamps = read_timestamp_file(left_timestamp_file)
    
    if not left_timestamps:
        print("No left timestamp data found")
        return
    
    print(f"Found {len(left_timestamps)} left camera timestamps")
    
    # Analyze frame timing consistency
    if len(left_timestamps) > 1:
        durations = []
        for i in range(len(left_timestamps) - 1):
            dt = left_timestamps[i+1][1] - left_timestamps[i][1]
            durations.append(dt)
        
        print(f"\nTiming analysis:")
        print(f"   Average frame interval: {np.mean(durations):.3f}s ({1/np.mean(durations):.1f} FPS)")
        print(f"   Min frame interval: {min(durations):.3f}s ({1/min(durations):.1f} FPS)")
        print(f"   Max frame interval: {max(durations):.3f}s ({1/max(durations):.1f} FPS)")
        print(f"   Standard deviation: {np.std(durations):.3f}s")
        
        # Check for timing consistency
        cv = np.std(durations) / np.mean(durations)  # coefficient of variation
        if cv < 0.1:
            print("   Timing is very consistent")
        elif cv < 0.2:
            print("   Timing is moderately consistent") 
        else:
            print("   Timing varies significantly")


def compare_timestamp_files_with_fps_data(timestamp_file, fps_data_file='fps_data.npy'):
    """
    Compare timestamp file data with your FPS measurements
    """
    try:
        fps_data = np.load(fps_data_file)
        print(f"Loaded {len(fps_data)} FPS measurements")
    except FileNotFoundError:
        print(f"FPS data file {fps_data_file} not found - skipping comparison")
        return
    
    timestamps = read_timestamp_file(timestamp_file)
    
    if len(timestamps) < 2:
        print("Need at least 2 frames for comparison")
        return
    
    # Calculate durations from timestamps
    timestamp_durations = []
    for i in range(len(timestamps) - 1):
        dt = timestamps[i+1][1] - timestamps[i][1]
        timestamp_durations.append(dt)
    
    # Convert FPS to durations
    fps_durations = [1.0 / max(0.1, fps) for fps in fps_data]
    
    # Compare
    min_len = min(len(timestamp_durations), len(fps_durations))
    
    print(f"\nComparison: Timestamp file vs FPS measurements (first 5 frames):")
    print("Frame | File Duration | FPS Duration | Difference | Which is faster?")
    print("-" * 70)
    
    for i in range(min(5, min_len)):
        file_dur = timestamp_durations[i]
        fps_dur = fps_durations[i]
        diff = abs(file_dur - fps_dur)
        
        if file_dur < fps_dur:
            faster = "File (actual)"
        else:
            faster = "FPS (measured)"
        
        print(f"{i:5d} | {file_dur:.3f}s      | {fps_dur:.3f}s     | {diff:.3f}s    | {faster}")
    
    if min_len > 0:
        avg_file_dur = np.mean(timestamp_durations[:min_len])
        avg_fps_dur = np.mean(fps_durations[:min_len])
        
        print(f"\nSummary:")
        print(f"   Average file duration: {avg_file_dur:.3f}s ({1/avg_file_dur:.1f} FPS)")
        print(f"   Average FPS duration:  {avg_fps_dur:.3f}s ({1/avg_fps_dur:.1f} FPS)")
        print(f"   Timestamp file represents ACTUAL capture timing")
        print(f"   FPS measurements represent processing speed")
        print(f"   Use timestamp file for most accurate video!")


def main():
    """
    Main function to analyze turtle frame timestamps using only left camera data
    """
    
    # Your project directory
    image_dir = '/home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/video/09_04_2025_NEA_video/04_16_2025_03_03_17_tetherless_A'
    
    # Left timestamp file only
    left_timestamp_file ='/home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/video/09_04_2025_NEA_video/04_16_2025_03_03_17_tetherless_A/timestamps.txt'
    
    print("Turtle Frame Timestamp Analysis (using left camera only)")
    print("=" * 55)
    
    # Check if left timestamp file exists
    if not os.path.exists(left_timestamp_file):
        print(f"Left timestamp file not found: {left_timestamp_file}")
        print("Please make sure the timestamp file exists in the correct location.")
        return
    
    # Analyze left camera timing
    analyze_left_timestamps(left_timestamp_file)
    
    # Create FFmpeg concat file using left timestamps
    print(f"\nCreating FFmpeg concat file using left camera timestamps...")
    concat_file = create_ffmpeg_concat_from_timestamp_file(
        image_dir, left_timestamp_file, "combined"
    )
    
    if concat_file:
        print(f"\nTo create video with accurate timing:")
        print(f"cd {image_dir}")
        concat_name = os.path.basename(concat_file)
        print(f"ffmpeg -f concat -safe 0 -i {concat_name} -c:v libx264 -pix_fmt yuv420p turtle_realtime.mp4")
        print(f"\nThis video will show your turtle's ACTUAL real-time performance!")
    else:
        print("Failed to create concat file.")
    
    # Compare with FPS measurements if available
    print("\nComparing with FPS measurements (if available)...")
    compare_timestamp_files_with_fps_data(left_timestamp_file)


if __name__ == "__main__":
    main()

Turtle Frame Timestamp Analysis (using left camera only)

Analyzing left camera timestamp data...
Found 527 left camera timestamps

Timing analysis:
   Average frame interval: 0.260s (3.8 FPS)
   Min frame interval: 0.181s (5.5 FPS)
   Max frame interval: 0.439s (2.3 FPS)
   Standard deviation: 0.052s
   Timing is moderately consistent

Creating FFmpeg concat file using left camera timestamps...
Reading timestamps from: /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/video/09_04_2025_NEA_video/04_16_2025_03_03_17_tetherless_A/timestamps.txt
Using frames from: combined

Analyzing timestamps from file: 527 entries...
First few timestamp entries:
  Frame 0: 2025-04-16 03:03:20.383 (Unix: 1744787000.383956)
  Frame 1: 2025-04-16 03:03:20.739 (Unix: 1744787000.739956)
  Frame 2: 2025-04-16 03:03:20.963 (Unix: 1744787000.963956)
  Frame 3: 2025-04-16 03:03:21.181 (Unix: 1744787001.181956)
  Frame 4: 2025-04-16 03:03:21.413 (Unix: 1744787001.413956)
Capture Analysis:
   Fi